In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("blinkit_orders.csv")
delivery = pd.read_csv("blinkit_delivery_performance.csv")
feedback = pd.read_csv("blinkit_customer_feedback.csv")
customers = pd.read_csv("blinkit_customers.csv")

In [3]:
print(orders.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   order_id                5000 non-null   int64  
 1   customer_id             5000 non-null   int64  
 2   order_date              5000 non-null   object 
 3   promised_delivery_time  5000 non-null   object 
 4   actual_delivery_time    5000 non-null   object 
 5   delivery_status         5000 non-null   object 
 6   order_total             5000 non-null   float64
 7   payment_method          5000 non-null   object 
 8   delivery_partner_id     5000 non-null   int64  
 9   store_id                5000 non-null   int64  
dtypes: float64(1), int64(4), object(5)
memory usage: 390.8+ KB
None


In [4]:
print(orders.head(5))

     order_id  customer_id           order_date promised_delivery_time  \
0  1961864118     30065862  2024-07-17 08:34:01    2024-07-17 08:52:01   
1  1549769649      9573071  2024-05-28 13:14:29    2024-05-28 13:25:29   
2  9185164487     45477575  2024-09-23 13:07:12    2024-09-23 13:25:12   
3  9644738826     88067569  2023-11-24 16:16:56    2023-11-24 16:34:56   
4  5427684290     83298567  2023-11-20 05:00:39    2023-11-20 05:17:39   

  actual_delivery_time delivery_status  order_total payment_method  \
0  2024-07-17 08:47:01         On Time      3197.07           Cash   
1  2024-05-28 13:27:29         On Time       976.55           Cash   
2  2024-09-23 13:29:12         On Time       839.05            UPI   
3  2023-11-24 16:33:56         On Time       440.23           Card   
4  2023-11-20 05:18:39         On Time      2526.68           Cash   

   delivery_partner_id  store_id  
0                63230      4771  
1                14983      7534  
2                39859      9

In [5]:
files = [
    "blinkit_orders.csv",
    "blinkit_delivery_performance.csv",
    "blinkit_customer_feedback.csv",
    "blinkit_customers.csv"
]

for f in files:
    try:
        df = pd.read_csv(f)
        print(f"================ {f} ================")
        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        print(df.head(2))
        print("\n")
    except Exception as e:
        print(f"Could not load {f}: {e}")

================ blinkit_orders.csv ================
Rows: 5000
Columns: ['order_id', 'customer_id', 'order_date', 'promised_delivery_time', 'actual_delivery_time', 'delivery_status', 'order_total', 'payment_method', 'delivery_partner_id', 'store_id']
     order_id  customer_id           order_date promised_delivery_time  \
0  1961864118     30065862  2024-07-17 08:34:01    2024-07-17 08:52:01   
1  1549769649      9573071  2024-05-28 13:14:29    2024-05-28 13:25:29   

  actual_delivery_time delivery_status  order_total payment_method  \
0  2024-07-17 08:47:01         On Time      3197.07           Cash   
1  2024-05-28 13:27:29         On Time       976.55           Cash   

   delivery_partner_id  store_id  
0                63230      4771  
1                14983      7534  


================ blinkit_delivery_performance.csv ================
Rows: 5000
Columns: ['order_id', 'delivery_partner_id', 'promised_time', 'actual_time', 'delivery_time_minutes', 'distance_km', 'delivery_st

In [6]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce') 

In [7]:
orders['promised_delivery_time'] = pd.to_datetime(orders['promised_delivery_time'], errors='coerce') 
orders['actual_delivery_time'] = pd.to_datetime(orders['actual_delivery_time'], errors='coerce') 

In [8]:
orders['actual_delivery_in_mins'] = (
    orders['actual_delivery_time'] - orders['order_date']
).dt.total_seconds() / 60

In [9]:
orders['promised_delivery_in_mins'] = (
    orders['promised_delivery_time'] - orders['order_date']
).dt.total_seconds() / 60

In [10]:
orders['is_delayed'] = (orders['actual_delivery_time'] > orders['promised_delivery_time']).astype(int)

In [11]:
print(orders.head(5))

     order_id  customer_id          order_date promised_delivery_time  \
0  1961864118     30065862 2024-07-17 08:34:01    2024-07-17 08:52:01   
1  1549769649      9573071 2024-05-28 13:14:29    2024-05-28 13:25:29   
2  9185164487     45477575 2024-09-23 13:07:12    2024-09-23 13:25:12   
3  9644738826     88067569 2023-11-24 16:16:56    2023-11-24 16:34:56   
4  5427684290     83298567 2023-11-20 05:00:39    2023-11-20 05:17:39   

  actual_delivery_time delivery_status  order_total payment_method  \
0  2024-07-17 08:47:01         On Time      3197.07           Cash   
1  2024-05-28 13:27:29         On Time       976.55           Cash   
2  2024-09-23 13:29:12         On Time       839.05            UPI   
3  2023-11-24 16:33:56         On Time       440.23           Card   
4  2023-11-20 05:18:39         On Time      2526.68           Cash   

   delivery_partner_id  store_id  actual_delivery_in_mins  \
0                63230      4771                     13.0   
1                1

In [12]:
df_analytics = orders.merge(
    delivery[['order_id', 'distance_km', 'reasons_if_delayed']],
    on ='order_id',
    how='left'
)

In [13]:
df_analytics = df_analytics.merge(
    feedback[['order_id', 'rating', 'feedback_category', 'sentiment']],
    on ='order_id',
    how='left'
)

In [14]:
df_analytics = df_analytics.merge(
    customers[['customer_id', 'area', 'customer_segment']],
    on ='customer_id',
    how='left'
)

In [15]:
print(df_analytics.head(10))

     order_id  customer_id          order_date promised_delivery_time  \
0  1961864118     30065862 2024-07-17 08:34:01    2024-07-17 08:52:01   
1  1549769649      9573071 2024-05-28 13:14:29    2024-05-28 13:25:29   
2  9185164487     45477575 2024-09-23 13:07:12    2024-09-23 13:25:12   
3  9644738826     88067569 2023-11-24 16:16:56    2023-11-24 16:34:56   
4  5427684290     83298567 2023-11-20 05:00:39    2023-11-20 05:17:39   
5  3265154092     43367112 2023-03-18 16:29:51    2023-03-18 16:49:51   
6  4898355547     13284996 2023-04-16 18:50:37    2023-04-16 19:01:37   
7  6568151549     88866835 2024-03-31 06:26:48    2024-03-31 06:37:48   
8  6006693867     24496983 2023-07-13 23:49:36    2023-07-14 00:02:36   
9   374186990     52215833 2023-08-09 01:17:30    2023-08-09 01:37:30   

  actual_delivery_time   delivery_status  order_total payment_method  \
0  2024-07-17 08:47:01           On Time      3197.07           Cash   
1  2024-05-28 13:27:29           On Time       976.5

In [20]:
df_analytics['reasons_if_delayed'] = df_analytics['reasons_if_delayed'].fillna('On Time / No Delay')

In [21]:
print(df_analytics.head(10))

     order_id  customer_id          order_date promised_delivery_time  \
0  1961864118     30065862 2024-07-17 08:34:01    2024-07-17 08:52:01   
1  1549769649      9573071 2024-05-28 13:14:29    2024-05-28 13:25:29   
2  9185164487     45477575 2024-09-23 13:07:12    2024-09-23 13:25:12   
3  9644738826     88067569 2023-11-24 16:16:56    2023-11-24 16:34:56   
4  5427684290     83298567 2023-11-20 05:00:39    2023-11-20 05:17:39   
5  3265154092     43367112 2023-03-18 16:29:51    2023-03-18 16:49:51   
6  4898355547     13284996 2023-04-16 18:50:37    2023-04-16 19:01:37   
7  6568151549     88866835 2024-03-31 06:26:48    2024-03-31 06:37:48   
8  6006693867     24496983 2023-07-13 23:49:36    2023-07-14 00:02:36   
9   374186990     52215833 2023-08-09 01:17:30    2023-08-09 01:37:30   

  actual_delivery_time   delivery_status  order_total payment_method  \
0  2024-07-17 08:47:01           On Time      3197.07           Cash   
1  2024-05-28 13:27:29           On Time       976.5

In [22]:
df_analytics['reasons_if_delayed'] = df_analytics['reasons_if_delayed'].replace('On Time / No Delay', 'No Delay')

In [23]:
print(df_analytics.head(10))

     order_id  customer_id          order_date promised_delivery_time  \
0  1961864118     30065862 2024-07-17 08:34:01    2024-07-17 08:52:01   
1  1549769649      9573071 2024-05-28 13:14:29    2024-05-28 13:25:29   
2  9185164487     45477575 2024-09-23 13:07:12    2024-09-23 13:25:12   
3  9644738826     88067569 2023-11-24 16:16:56    2023-11-24 16:34:56   
4  5427684290     83298567 2023-11-20 05:00:39    2023-11-20 05:17:39   
5  3265154092     43367112 2023-03-18 16:29:51    2023-03-18 16:49:51   
6  4898355547     13284996 2023-04-16 18:50:37    2023-04-16 19:01:37   
7  6568151549     88866835 2024-03-31 06:26:48    2024-03-31 06:37:48   
8  6006693867     24496983 2023-07-13 23:49:36    2023-07-14 00:02:36   
9   374186990     52215833 2023-08-09 01:17:30    2023-08-09 01:37:30   

  actual_delivery_time   delivery_status  order_total payment_method  \
0  2024-07-17 08:47:01           On Time      3197.07           Cash   
1  2024-05-28 13:27:29           On Time       976.5

In [24]:
df_analytics.to_csv("blinkit_cleaned.csv", index=False)